# INR/USD Unified Error Correction Model

Tests whether deviations from the composite REER+FEER equilibrium predict future INR returns.
The model operates in three tiers:

1. **Composite equilibrium** — 50/50 blend of REER fair and FEER static fair (publication-lag-adjusted)
2. **Multi-horizon predictability** — testing 1 to 12-month forward returns against the ECT
3. **Regime conditioning** — splitting the error correction coefficient by calm vs stress regimes

Requires output files from `05_data_and_models.ipynb`.


## 1. Composite Equilibrium + ADF Gate


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from statsmodels.tsa.stattools import adfuller

PROCESSED = '../data/processed/'

ml = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
ml.index = pd.to_datetime(ml.index).to_period('M').to_timestamp()
p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()
p3 = pd.read_csv(PROCESSED+'phase3_feer_v2.csv', index_col=0, parse_dates=True)
p3.index = pd.to_datetime(p3.index).to_period('M').to_timestamp()
p4 = pd.read_csv(PROCESSED+'phase4_beer_v2.csv', index_col=0, parse_dates=True)
p4.index = pd.to_datetime(p4.index).to_period('M').to_timestamp()

# REER fair is monthly and real-time; FEER fair is quarterly with 6-month publication lag
reer_fair = p2['inr_fair_reer'].dropna()
feer_shifted = p3['inr_fair_static'].dropna().copy()
feer_shifted.index = feer_shifted.index + pd.DateOffset(months=6)
feer_monthly = feer_shifted.reindex(pd.date_range(feer_shifted.index.min(), ml.index.max(), freq='MS')).ffill()

# composite: average where both available, REER-only elsewhere
feer_avail = feer_monthly.dropna()
composite = pd.Series(dtype=float, index=reer_fair.index)
for dt in reer_fair.index:
    r = reer_fair[dt]
    composite[dt] = 0.5*r + 0.5*feer_avail[dt] if dt in feer_avail.index else r

# ECT = log(actual) - log(fair)   positive = INR weaker than fair
inr = ml['inr_usd'].reindex(composite.dropna().index).dropna()
common = inr.index.intersection(composite.dropna().index)
ecm = pd.DataFrame({'inr_usd': inr.loc[common], 'composite_fair': composite.loc[common],
                     'reer_fair': reer_fair.reindex(common), 'feer_fair': feer_monthly.reindex(common)},
                    index=common).dropna()
ecm['log_inr']  = np.log(ecm['inr_usd'])
ecm['log_fair'] = np.log(ecm['composite_fair'])
ecm['ect']      = ecm['log_inr'] - ecm['log_fair']
ecm['p_stress'] = p4['p_stress'].reindex(ecm.index)

# pre-build drivers for the ECM regression
ecm['d_log_inr']   = ecm['log_inr'].diff()
ecm['d_log_dxy']   = np.log(ml['dxy']).reindex(ecm.index).diff()
ecm['d_log_brent'] = np.log(ml['brent']).reindex(ecm.index).diff()
ecm['d_rrd']       = ml['real_rate_diff'].reindex(ecm.index).diff()
ecm['d_vix']       = ml['vix'].reindex(ecm.index).diff()
ecm['d_fpi']       = ml['fpi_pct_gdp'].reindex(ecm.index).diff()
ecm['ect_lag1']    = ecm['ect'].shift(1)

# stationarity gate
adf = adfuller(ecm['ect'].dropna(), maxlag=12, autolag='AIC')
print(f"ADF on ECT: stat={adf[0]:.3f}, p={adf[1]:.4f}")
print(f"{'GO' if adf[1]<0.05 else 'STOP'} — ECT is {'stationary' if adf[1]<0.05 else 'NOT stationary'}")

e = ecm['ect']
print(f"\nECT: mean={e.mean():+.4f}, std={e.std():.4f}, current={e.iloc[-1]:+.4f} ({abs(e.iloc[-1])*100:.1f}% {'weak' if e.iloc[-1]>0 else 'strong'})")

ecm.to_csv(PROCESSED+'ecm_base.csv')


## 2. Multi-Horizon Predictability + Regime Conditioning

Monthly α is expected to be insignificant (Meese-Rogoff). The real test is whether longer-horizon
forward returns respond to the ECT, and whether the response differs across calm vs stress regimes.


In [ ]:
import statsmodels.api as sm

ecm = pd.read_csv(PROCESSED+'ecm_base.csv', index_col=0, parse_dates=True)
ecm.index = pd.to_datetime(ecm.index).to_period('M').to_timestamp()

# --- multi-horizon ---
print(f"{'h':>4}{'α':>10}{'t':>7}{'p':>7}{'R²':>7}")
for h in [1, 3, 6, 9, 12]:
    fwd = ecm['log_inr'].shift(-h) - ecm['log_inr']
    dd = pd.DataFrame({'fwd': fwd, 'ect': ecm['ect']}).dropna()
    r = sm.OLS(dd['fwd'], sm.add_constant(dd['ect'])).fit(cov_type='HAC', cov_kwds={'maxlags':max(h,6)})
    sig = '***' if r.pvalues['ect']<0.01 else ('**' if r.pvalues['ect']<0.05 else ('*' if r.pvalues['ect']<0.1 else ''))
    print(f"{h:>4}{r.params['ect']:>+10.4f}{r.tvalues['ect']:>7.2f}{r.pvalues['ect']:>7.3f}{r.rsquared:>7.3f} {sig}")

# --- regime conditioning at 1-month ---
d3 = ecm[['d_log_inr','ect_lag1','p_stress','d_log_dxy','d_log_brent']].dropna()
d3['ect_calm']   = d3['ect_lag1'] * (1 - d3['p_stress'])
d3['ect_stress'] = d3['ect_lag1'] * d3['p_stress']
r3 = sm.OLS(d3['d_log_inr'], sm.add_constant(d3[['ect_calm','ect_stress','d_log_dxy','d_log_brent']])).fit(
    cov_type='HAC', cov_kwds={'maxlags':6})
print(f"\n1-month regime split: α_calm={r3.params['ect_calm']:+.4f} (p={r3.pvalues['ect_calm']:.3f}), α_stress={r3.params['ect_stress']:+.4f} (p={r3.pvalues['ect_stress']:.3f})")

# --- regime conditioning at 6-month ---
fwd6 = ecm['log_inr'].shift(-6) - ecm['log_inr']
d6 = pd.DataFrame({'fwd6': fwd6, 'ect': ecm['ect'], 'p_stress': ecm['p_stress']}).dropna()
d6['ect_calm']   = d6['ect'] * (1 - d6['p_stress'])
d6['ect_stress'] = d6['ect'] * d6['p_stress']
r6 = sm.OLS(d6['fwd6'], sm.add_constant(d6[['ect_calm','ect_stress']])).fit(
    cov_type='HAC', cov_kwds={'maxlags':6})
print(f"6-month regime split: α_calm={r6.params['ect_calm']:+.4f} (p={r6.pvalues['ect_calm']:.3f}), α_stress={r6.params['ect_stress']:+.4f} (p={r6.pvalues['ect_stress']:.3f})")

ecm['fwd_1m']  = ecm['log_inr'].shift(-1) - ecm['log_inr']
ecm['fwd_6m']  = ecm['log_inr'].shift(-6) - ecm['log_inr']
ecm['fwd_12m'] = ecm['log_inr'].shift(-12) - ecm['log_inr']
ecm.to_csv(PROCESSED+'ecm_base.csv')


## 3. Out-of-Sample Signal Test

Train the 12-month ECT model on pre-2019 data, predict forward returns out of sample,
then run an expanding-window test to verify α stability.


In [ ]:
ecm = pd.read_csv(PROCESSED+'ecm_base.csv', index_col=0, parse_dates=True)
ecm.index = pd.to_datetime(ecm.index).to_period('M').to_timestamp()

SPLIT = pd.Timestamp('2019-01-01')
train = ecm[ecm.index < SPLIT].dropna(subset=['ect','fwd_12m'])
test  = ecm[ecm.index >= SPLIT].dropna(subset=['ect','fwd_12m'])

is_model = sm.OLS(train['fwd_12m'], sm.add_constant(train['ect'])).fit(cov_type='HAC', cov_kwds={'maxlags':12})
print(f"In-sample: α={is_model.params['ect']:+.4f} (p={is_model.pvalues['ect']:.4f}), R²={is_model.rsquared:.3f}")

test = test.copy()
test['predicted_12m'] = is_model.predict(sm.add_constant(test['ect']))
test['correct'] = (test['predicted_12m'] < 0) == (test['fwd_12m'] < 0)
print(f"OOS directional accuracy: {test['correct'].sum()}/{len(test)} = {test['correct'].mean()*100:.1f}%")
print(f"OOS correlation: Pearson={test['predicted_12m'].corr(test['fwd_12m']):.3f}, Spearman={test['predicted_12m'].corr(test['fwd_12m'], method='spearman'):.3f}")

# expanding window to check α stability
min_train = 120
results = []
for i in range(min_train, len(ecm)):
    dt = ecm.index[i]
    if pd.isna(ecm.loc[dt, 'fwd_12m']): continue
    tr = ecm.iloc[:i].dropna(subset=['ect','fwd_12m'])
    if len(tr) < min_train: continue
    try:
        mod = sm.OLS(tr['fwd_12m'], sm.add_constant(tr['ect'])).fit()
        results.append({'date': dt, 'alpha': mod.params['ect']})
    except: continue

oos_alpha = pd.DataFrame(results).set_index('date')['alpha']
print(f"\nRolling α: mean={oos_alpha.mean():.3f}, range=[{oos_alpha.min():.3f}, {oos_alpha.max():.3f}]")
print(f"{'Never flips sign' if oos_alpha.max() < 0 else 'Flips sign in some windows'}")

ecm.to_csv(PROCESSED+'ecm_base.csv')


## 4. Summary Dashboard


In [ ]:
ecm = pd.read_csv(PROCESSED+'ecm_base.csv', index_col=0, parse_dates=True)
ecm.index = pd.to_datetime(ecm.index).to_period('M').to_timestamp()
p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()
p3 = pd.read_csv(PROCESSED+'phase3_feer_v2.csv', index_col=0, parse_dates=True)
p3.index = pd.to_datetime(p3.index).to_period('M').to_timestamp()
p4 = pd.read_csv(PROCESSED+'phase4_beer_v2.csv', index_col=0, parse_dates=True)
p4.index = pd.to_datetime(p4.index).to_period('M').to_timestamp()
ml = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
ml.index = pd.to_datetime(ml.index).to_period('M').to_timestamp()

r2, r3, r4 = p2.iloc[-1], p3.iloc[-1], p4.iloc[-1]
re = ecm.dropna(subset=['ect']).iloc[-1]
p_stress = p4['p_stress'].dropna().iloc[-1]

print("INR/USD HYBRID FAIR VALUE MODEL")
print(f"Spot: {re['inr_usd']:.2f} ({ecm.dropna(subset=['ect']).index[-1].strftime('%b %Y')})\n")

print(f"{'Model':<28}{'Misalign':>10}{'Fair INR':>10}")
print("-" * 50)
print(f"{'REER structural':<28}{r2['misalignment_pct']:>+9.1f}%{r2['inr_fair_reer']:>10.2f}")
print(f"{'FEER static':<28}{r3['feer_static_pct']:>+9.1f}%{r3['inr_fair_static']:>10.2f}")
print(f"{'FEER conditional':<28}{r3['feer_cond_pct']:>+9.1f}%{r3['inr_fair_cond']:>10.2f}")
print(f"{'BEER short-run':<28}{r4['beer_misalign']:>+9.1f}%{r4['inr_fair_beer']:>10.2f}")
print(f"{'Composite ECM':<28}{re['ect']*100:>+9.1f}%{re['composite_fair']:>10.2f}")
print(f"\nFinancing premium: {r3['financing_premium']:+.1f}pp")
print(f"P(stress): {p_stress:.2f} [{'STRESS' if p_stress > 0.5 else 'CALM'}]")
print(f"Brent: ${ml['brent'].dropna().iloc[-1]:.0f} (norm $72)")

# ECM signal
alpha_12, const_12 = -0.450, 0.038
pred = const_12 + alpha_12 * re['ect']
print(f"\n12-month prediction: {pred*100:+.1f}% (trend {const_12*100:+.1f}% + reversion {alpha_12*re['ect']*100:+.1f}%)")
if abs(pred) < 0.02:   print("Forces roughly balance — INR approximately STABLE")
elif pred < 0:          print("Reversion dominates — INR APPRECIATION expected")
else:                   print("Trend dominates — continued DEPRECIATION expected")

if p_stress > 0.5:
    print(f"\nSTRESS REGIME ACTIVE — reversion historically unreliable. Wait for calm.")
